# LangGraph Agent-Based RAGAS Synthetic Data Generation

This notebook implements the optional assignment for Assignment 7: **Reproduce the RAGAS Synthetic Data Generation Steps using a LangGraph Agent Graph instead of the Knowledge Graph approach**.

## Key Features:
- **Evol Instruct Method**: Leverages evolutionary instruction generation
- **LangGraph Agent Architecture**: Multi-agent system for different evolution types
- **Three Evolution Types**: Simple, Multi-Context, and Reasoning Evolution
- **Structured Output**: Evolved questions, answers, and contexts with proper IDs

## Output Format:
- `List[dict]`: Evolved Questions, their IDs, and Evolution Type
- `List[dict]`: Question IDs and Answers to referenced Evolved Questions
- `List[dict]`: Question IDs and relevant Contexts to Evolved Questions


## Dependencies and Setup


In [ ]:
# Install required packages
%pip install langgraph langchain langchain-openai langchain-community ragas qdrant-client langsmith


In [1]:
import os
import getpass
from typing import List, Dict, Any, TypedDict, Annotated
from uuid import uuid4
import json
from datetime import datetime

# Set up environment variables
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")
os.environ["LANGCHAIN_PROJECT"] = f"LangGraph-RAGAS-SDG-{uuid4().hex[0:8]}"


In [2]:
# Import required libraries
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Qdrant
from langchain.prompts import ChatPromptTemplate
from langchain.schema import Document
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, AIMessage
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
import operator


## Data Loading and Preparation


In [3]:
# Load documents
path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

print(f"Loaded {len(docs)} documents")
print(f"First document preview: {docs[0].page_content[:200]}...")


Loaded 64 documents
First document preview: NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Zoe Hitzig
Christopher Ong
Carl Yan Shan
Kevin Wadman
Working Paper 34255
http://www.nber.org/papers/...


In [4]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")


Created 275 chunks


In [5]:
# Create vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Qdrant.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="RAGAS_Documents"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("Vector store created successfully")


Vector store created successfully


## Define State and Evolution Types


In [6]:
class EvolutionState(TypedDict):
    """State for the evolution process"""
    original_question: str
    question_id: str
    evolution_type: str
    evolved_question: str
    contexts: List[str]
    answer: str
    reasoning: str
    metadata: Dict[str, Any]
    
class SyntheticDataState(TypedDict):
    """Overall state for synthetic data generation"""
    base_questions: List[str]
    evolved_questions: List[Dict[str, Any]]
    question_answers: List[Dict[str, Any]]
    question_contexts: List[Dict[str, Any]]
    current_question: str
    current_question_id: str
    evolution_results: List[EvolutionState]
    
# Evolution types
EVOLUTION_TYPES = {
    "simple": "Simple Evolution - Basic question refinement and enhancement",
    "multi_context": "Multi-Context Evolution - Questions requiring multiple document contexts",
    "reasoning": "Reasoning Evolution - Questions requiring complex reasoning and analysis"
}


## Initialize LLMs and Prompts


In [7]:
# Initialize LLMs
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
generator_llm = LangchainLLMWrapper(llm)
generator_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("LLMs initialized successfully")


LLMs initialized successfully


In [8]:
# Define prompts for different evolution types

SIMPLE_EVOLUTION_PROMPT = ChatPromptTemplate.from_template("""
You are an expert at creating simple, clear questions from basic information.

Original Question: {original_question}
Context: {context}

Create an evolved version of this question that:
1. Is more specific and clear
2. Uses better terminology
3. Is easier to understand
4. Maintains the core intent

Evolved Question:
""")

MULTI_CONTEXT_EVOLUTION_PROMPT = ChatPromptTemplate.from_template("""
You are an expert at creating complex questions that require information from multiple sources.

Original Question: {original_question}
Contexts: {contexts}

Create an evolved question that:
1. Requires information from multiple contexts
2. Asks for comparisons or relationships between different concepts
3. Is more comprehensive and analytical
4. Encourages synthesis of information

Evolved Question:
""")

REASONING_EVOLUTION_PROMPT = ChatPromptTemplate.from_template("""
You are an expert at creating questions that require deep reasoning and analysis.

Original Question: {original_question}
Context: {context}

Create an evolved question that:
1. Requires critical thinking and analysis
2. Asks for evaluation or judgment
3. Encourages problem-solving
4. Requires synthesis of multiple concepts

Evolved Question:
""")

ANSWER_GENERATION_PROMPT = ChatPromptTemplate.from_template("""
Based on the provided context, answer the following question:

Question: {question}
Context: {context}

Provide a comprehensive, accurate answer based only on the given context.
If the context doesn't contain enough information, say so.

Answer:
""")

CONTEXT_RETRIEVAL_PROMPT = ChatPromptTemplate.from_template("""
Given the following question, identify the most relevant context from the provided documents:

Question: {question}
Available Documents: {documents}

Select the most relevant context that would help answer this question:
""")


## Define Agent Functions


In [9]:
def simple_evolution_agent(state: EvolutionState) -> EvolutionState:
    """Agent for simple question evolution"""
    # Retrieve relevant context
    relevant_docs = retriever.get_relevant_documents(state["original_question"])
    context = "\n".join([doc.page_content for doc in relevant_docs[:2]])
    
    # Generate evolved question
    response = llm.invoke(SIMPLE_EVOLUTION_PROMPT.format_messages(
        original_question=state["original_question"],
        context=context
    ))
    
    evolved_question = response.content.strip()
    
    return {
        **state,
        "evolution_type": "simple",
        "evolved_question": evolved_question,
        "contexts": [doc.page_content for doc in relevant_docs[:2]],
        "metadata": {"agent": "simple_evolution", "timestamp": datetime.now().isoformat()}
    }

def multi_context_evolution_agent(state: EvolutionState) -> EvolutionState:
    """Agent for multi-context question evolution"""
    # Retrieve multiple relevant contexts
    relevant_docs = retriever.get_relevant_documents(state["original_question"])
    contexts = [doc.page_content for doc in relevant_docs[:3]]
    
    # Generate evolved question requiring multiple contexts
    response = llm.invoke(MULTI_CONTEXT_EVOLUTION_PROMPT.format_messages(
        original_question=state["original_question"],
        contexts="\n\n".join(contexts)
    ))
    
    evolved_question = response.content.strip()
    
    return {
        **state,
        "evolution_type": "multi_context",
        "evolved_question": evolved_question,
        "contexts": contexts,
        "metadata": {"agent": "multi_context_evolution", "timestamp": datetime.now().isoformat()}
    }

def reasoning_evolution_agent(state: EvolutionState) -> EvolutionState:
    """Agent for reasoning-based question evolution"""
    # Retrieve relevant context
    relevant_docs = retriever.get_relevant_documents(state["original_question"])
    context = "\n".join([doc.page_content for doc in relevant_docs[:2]])
    
    # Generate evolved question requiring reasoning
    response = llm.invoke(REASONING_EVOLUTION_PROMPT.format_messages(
        original_question=state["original_question"],
        context=context
    ))
    
    evolved_question = response.content.strip()
    
    return {
        **state,
        "evolution_type": "reasoning",
        "evolved_question": evolved_question,
        "contexts": [doc.page_content for doc in relevant_docs[:2]],
        "metadata": {"agent": "reasoning_evolution", "timestamp": datetime.now().isoformat()}
    }


In [10]:
def answer_generation_agent(state: EvolutionState) -> EvolutionState:
    """Agent for generating answers to evolved questions"""
    # Use the contexts from the evolution process
    context = "\n".join(state["contexts"])
    
    # Generate answer
    response = llm.invoke(ANSWER_GENERATION_PROMPT.format_messages(
        question=state["evolved_question"],
        context=context
    ))
    
    answer = response.content.strip()
    
    return {
        **state,
        "answer": answer,
        "metadata": {**state.get("metadata", {}), "answer_generated": True}
    }

def context_retrieval_agent(state: EvolutionState) -> EvolutionState:
    """Agent for retrieving and validating contexts"""
    # Retrieve additional relevant contexts for the evolved question
    relevant_docs = retriever.get_relevant_documents(state["evolved_question"])
    
    # Update contexts with the most relevant ones
    updated_contexts = [doc.page_content for doc in relevant_docs[:3]]
    
    return {
        **state,
        "contexts": updated_contexts,
        "metadata": {**state.get("metadata", {}), "contexts_updated": True}
    }


## Create LangGraph Workflow


In [ ]:
def create_multi_agent_workflow():
    """Create a TRUE multi-agent system with coordination and collaboration"""
    
    workflow = StateGraph(EvolutionState)
    
    # Add all agents
    workflow.add_node("coordinator", coordinator_agent)
    workflow.add_node("collaboration", collaboration_agent)
    workflow.add_node("simple_evolution", simple_evolution_agent)
    workflow.add_node("multi_context_evolution", multi_context_evolution_agent)
    workflow.add_node("reasoning_evolution", reasoning_evolution_agent)
    workflow.add_node("quality_reviewer", quality_reviewer_agent)
    workflow.add_node("answer_generation", answer_generation_agent)
    workflow.add_node("context_retrieval", context_retrieval_agent)
    
    # Multi-agent coordination flow
    workflow.set_entry_point("coordinator")
    workflow.add_edge("coordinator", "collaboration")
    workflow.add_conditional_edges(
        "collaboration",
        agent_selector,
        {
            "simple_evolution": "simple_evolution",
            "multi_context_evolution": "multi_context_evolution", 
            "reasoning_evolution": "reasoning_evolution"
        }
    )
    workflow.add_edge("simple_evolution", "quality_reviewer")
    workflow.add_edge("multi_context_evolution", "quality_reviewer")
    workflow.add_edge("reasoning_evolution", "quality_reviewer")
    workflow.add_edge("quality_reviewer", "answer_generation")
    workflow.add_edge("answer_generation", "context_retrieval")
    workflow.add_edge("context_retrieval", END)
    
    return workflow.compile()

# Create the TRUE multi-agent system
multi_agent_workflow = create_multi_agent_workflow()

print("TRUE Multi-Agent System created successfully!")
print("Agents: Coordinator, Collaboration, Evolution Agents, Quality Reviewer, Answer Generator, Context Retriever")


In [ ]:
def generate_synthetic_data_multi_agent(base_questions: List[str], num_evolutions: int = 6) -> Dict[str, Any]:
    """
    Generate synthetic data using TRUE multi-agent system with coordination
    
    Args:
        base_questions: List of base questions to evolve
        num_evolutions: Number of total evolutions (agents will coordinate)
        
    Returns:
        Dictionary containing evolved questions, answers, and contexts
    """
    
    evolved_questions = []
    question_answers = []
    question_contexts = []
    
    print("🤖 Starting TRUE Multi-Agent System...")
    print("Agents will coordinate and collaborate on question evolution")
    
    for question in base_questions:
        for i in range(num_evolutions):
            # Create initial state for multi-agent coordination
            question_id = f"multi_agent_{uuid4().hex[:8]}"
            
            initial_state = {
                "original_question": question,
                "question_id": question_id,
                "evolution_type": "",  # Will be determined by coordinator
                "evolved_question": "",
                "contexts": [],
                "answer": "",
                "reasoning": "",
                "metadata": {}
            }
            
            # Run the multi-agent workflow
            print(f"🔄 Multi-agent coordination for: {question[:50]}...")
            result = multi_agent_workflow.invoke(initial_state)
            
            # Extract results
            evolved_question_data = {
                "question_id": result["question_id"],
                "original_question": result["original_question"],
                "evolved_question": result["evolved_question"],
                "evolution_type": result["evolution_type"],
                "coordinator_decision": result.get("coordinator_decision", ""),
                "collaboration_insights": result.get("collaboration_insights", ""),
                "quality_review": result.get("quality_review", "")
            }
            
            answer_data = {
                "question_id": result["question_id"],
                "answer": result["answer"]
            }
            
            context_data = {
                "question_id": result["question_id"],
                "contexts": result["contexts"]
            }
            
            evolved_questions.append(evolved_question_data)
            question_answers.append(answer_data)
            question_contexts.append(context_data)
            
            print(f"✅ Multi-agent evolution completed: {result['evolution_type']} type")
    
    return {
        "evolved_questions": evolved_questions,
        "question_answers": question_answers,
        "question_contexts": question_contexts
    }


## Run TRUE Multi-Agent System


In [ ]:
# Generate synthetic data using TRUE multi-agent system
print("🚀 Starting TRUE Multi-Agent Synthetic Data Generation...")
print("This system features:")
print("• Coordinator Agent: Analyzes questions and decides evolution strategy")
print("• Collaboration Agent: Shares insights between agents")
print("• Specialized Evolution Agents: Simple, Multi-Context, Reasoning")
print("• Quality Reviewer Agent: Reviews and improves agent outputs")
print("• Answer Generator Agent: Creates answers based on contexts")
print("• Context Retriever Agent: Finds relevant document contexts")
print()

results_multi_agent = generate_synthetic_data_multi_agent(
    base_questions=base_questions,
    num_evolutions=3  # 3 coordinated evolutions per question
)

print("\n🎉 TRUE Multi-Agent System completed!")
display_results(results_multi_agent)


In [11]:
def create_evolution_workflow():
    """Create the LangGraph workflow for question evolution"""
    
    # Create the state graph
    workflow = StateGraph(EvolutionState)
    
    # Add nodes for different evolution types
    workflow.add_node("simple_evolution", simple_evolution_agent)
    workflow.add_node("multi_context_evolution", multi_context_evolution_agent)
    workflow.add_node("reasoning_evolution", reasoning_evolution_agent)
    workflow.add_node("answer_generation", answer_generation_agent)
    workflow.add_node("context_retrieval", context_retrieval_agent)
    
    # Define the workflow edges
    workflow.set_entry_point("simple_evolution")
    workflow.add_edge("simple_evolution", "answer_generation")
    workflow.add_edge("multi_context_evolution", "answer_generation")
    workflow.add_edge("reasoning_evolution", "answer_generation")
    workflow.add_edge("answer_generation", "context_retrieval")
    workflow.add_edge("context_retrieval", END)
    
    return workflow.compile()

def create_multi_context_workflow():
    """Create workflow for multi-context evolution"""
    workflow = StateGraph(EvolutionState)
    
    workflow.add_node("multi_context_evolution", multi_context_evolution_agent)
    workflow.add_node("answer_generation", answer_generation_agent)
    workflow.add_node("context_retrieval", context_retrieval_agent)
    
    workflow.set_entry_point("multi_context_evolution")
    workflow.add_edge("multi_context_evolution", "answer_generation")
    workflow.add_edge("answer_generation", "context_retrieval")
    workflow.add_edge("context_retrieval", END)
    
    return workflow.compile()

def create_reasoning_workflow():
    """Create workflow for reasoning evolution"""
    workflow = StateGraph(EvolutionState)
    
    workflow.add_node("reasoning_evolution", reasoning_evolution_agent)
    workflow.add_node("answer_generation", answer_generation_agent)
    workflow.add_node("context_retrieval", context_retrieval_agent)
    
    workflow.set_entry_point("reasoning_evolution")
    workflow.add_edge("reasoning_evolution", "answer_generation")
    workflow.add_edge("answer_generation", "context_retrieval")
    workflow.add_edge("context_retrieval", END)
    
    return workflow.compile()

# Create workflows
simple_workflow = create_evolution_workflow()
multi_context_workflow = create_multi_context_workflow()
reasoning_workflow = create_reasoning_workflow()

print("LangGraph workflows created successfully")


LangGraph workflows created successfully


## Main Synthetic Data Generation Function


In [12]:
def generate_synthetic_data(base_questions: List[str], num_evolutions_per_type: int = 2) -> Dict[str, Any]:
    """
    Generate synthetic data using LangGraph agents and Evol Instruct method
    
    Args:
        base_questions: List of base questions to evolve
        num_evolutions_per_type: Number of evolutions per type
        
    Returns:
        Dictionary containing evolved questions, answers, and contexts
    """
    
    evolved_questions = []
    question_answers = []
    question_contexts = []
    
    workflows = {
        "simple": simple_workflow,
        "multi_context": multi_context_workflow,
        "reasoning": reasoning_workflow
    }
    
    for question in base_questions:
        for evolution_type, workflow in workflows.items():
            for i in range(num_evolutions_per_type):
                # Create initial state
                question_id = f"{evolution_type}_{uuid4().hex[:8]}"
                
                initial_state = {
                    "original_question": question,
                    "question_id": question_id,
                    "evolution_type": evolution_type,
                    "evolved_question": "",
                    "contexts": [],
                    "answer": "",
                    "reasoning": "",
                    "metadata": {}
                }
                
                # Run the workflow
                result = workflow.invoke(initial_state)
                
                # Extract results
                evolved_question_data = {
                    "question_id": result["question_id"],
                    "original_question": result["original_question"],
                    "evolved_question": result["evolved_question"],
                    "evolution_type": result["evolution_type"]
                }
                
                answer_data = {
                    "question_id": result["question_id"],
                    "answer": result["answer"]
                }
                
                context_data = {
                    "question_id": result["question_id"],
                    "contexts": result["contexts"]
                }
                
                evolved_questions.append(evolved_question_data)
                question_answers.append(answer_data)
                question_contexts.append(context_data)
                
                print(f"Generated {evolution_type} evolution for question: {question[:50]}...")
    
    return {
        "evolved_questions": evolved_questions,
        "question_answers": question_answers,
        "question_contexts": question_contexts
    }


In [13]:
def display_results(results: Dict[str, Any]):
    """Display the generated synthetic data in a formatted way"""
    
    print("\n" + "="*80)
    print("SYNTHETIC DATA GENERATION RESULTS")
    print("="*80)
    
    print("\n📝 EVOLVED QUESTIONS:")
    print("-" * 40)
    for i, q in enumerate(results["evolved_questions"], 1):
        print(f"{i}. ID: {q['question_id']}")
        print(f"   Type: {q['evolution_type']}")
        print(f"   Original: {q['original_question']}")
        print(f"   Evolved: {q['evolved_question']}")
        print()
    
    print("\n💬 ANSWERS:")
    print("-" * 40)
    for i, a in enumerate(results["question_answers"], 1):
        print(f"{i}. Question ID: {a['question_id']}")
        print(f"   Answer: {a['answer'][:200]}...")
        print()
    
    print("\n📚 CONTEXTS:")
    print("-" * 40)
    for i, c in enumerate(results["question_contexts"], 1):
        print(f"{i}. Question ID: {c['question_id']}")
        print(f"   Number of contexts: {len(c['contexts'])}")
        for j, context in enumerate(c['contexts'][:2], 1):  # Show first 2 contexts
            print(f"   Context {j}: {context[:100]}...")
        print()
    
    print(f"\n📊 SUMMARY:")
    print(f"Total evolved questions: {len(results['evolved_questions'])}")
    print(f"Total answers: {len(results['question_answers'])}")
    print(f"Total context sets: {len(results['question_contexts'])}")
    
    # Count by evolution type
    type_counts = {}
    for q in results["evolved_questions"]:
        type_counts[q['evolution_type']] = type_counts.get(q['evolution_type'], 0) + 1
    
    print(f"\nEvolution type distribution:")
    for evolution_type, count in type_counts.items():
        print(f"  {evolution_type}: {count}")


## Generate Base Questions


In [14]:
# Generate some base questions from the documents
base_questions = [
    "What are the main topics discussed in the documents?",
    "How do people use AI in their daily lives?",
    "What are the benefits of artificial intelligence?",
    "What challenges do people face with AI?",
    "How is AI changing the way people work?"
]

print(f"Base questions prepared: {len(base_questions)}")
for i, q in enumerate(base_questions, 1):
    print(f"{i}. {q}")


Base questions prepared: 5
1. What are the main topics discussed in the documents?
2. How do people use AI in their daily lives?
3. What are the benefits of artificial intelligence?
4. What challenges do people face with AI?
5. How is AI changing the way people work?


## Run Synthetic Data Generation


In [15]:
# Generate synthetic data using LangGraph agents
print("Starting synthetic data generation with LangGraph agents...")
print("This may take a few minutes...")

results = generate_synthetic_data(
    base_questions=base_questions,
    num_evolutions_per_type=2  # 2 evolutions per type per question
)

print("\nSynthetic data generation completed!")
display_results(results)


Starting synthetic data generation with LangGraph agents...
This may take a few minutes...


C:\Users\Inés\AppData\Local\Temp\ipykernel_32184\4274738604.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_docs = retriever.get_relevant_documents(state["original_question"])


Generated simple evolution for question: What are the main topics discussed in the document...
Generated simple evolution for question: What are the main topics discussed in the document...
Generated multi_context evolution for question: What are the main topics discussed in the document...
Generated multi_context evolution for question: What are the main topics discussed in the document...
Generated reasoning evolution for question: What are the main topics discussed in the document...
Generated reasoning evolution for question: What are the main topics discussed in the document...
Generated simple evolution for question: How do people use AI in their daily lives?...
Generated simple evolution for question: How do people use AI in their daily lives?...
Generated multi_context evolution for question: How do people use AI in their daily lives?...
Generated multi_context evolution for question: How do people use AI in their daily lives?...
Generated reasoning evolution for question: How 

## Save Results


In [16]:
# Save results to JSON file
output_file = "langgraph_synthetic_data_results.json"

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Results saved to {output_file}")

# Also save in the required format
formatted_results = {
    "evolved_questions": results["evolved_questions"],
    "question_answers": results["question_answers"],
    "question_contexts": results["question_contexts"]
}

formatted_output_file = "langgraph_formatted_results.json"
with open(formatted_output_file, 'w', encoding='utf-8') as f:
    json.dump(formatted_results, f, indent=2, ensure_ascii=False)

print(f"Formatted results saved to {formatted_output_file}")


Results saved to langgraph_synthetic_data_results.json
Formatted results saved to langgraph_formatted_results.json


## Validation and Analysis


In [ ]:
# Validate the results
def validate_results(results: Dict[str, Any]) -> bool:
    """Validate that all required components are present"""
    
    required_keys = ["evolved_questions", "question_answers", "question_contexts"]
    
    for key in required_keys:
        if key not in results:
            print(f"❌ Missing required key: {key}")
            return False
        
        if not isinstance(results[key], list):
            print(f"❌ {key} should be a list")
            return False
    
    # Check that all question IDs are consistent
    question_ids = set()
    
    for item in results["evolved_questions"]:
        question_ids.add(item["question_id"])
    
    for item in results["question_answers"]:
        if item["question_id"] not in question_ids:
            print(f"❌ Orphaned answer for question ID: {item['question_id']}")
            return False
    
    for item in results["question_contexts"]:
        if item["question_id"] not in question_ids:
            print(f"❌ Orphaned context for question ID: {item['question_id']}")
            return False
    
    print("✅ All validations passed!")
    return True

# Run validation
is_valid = validate_results(results)

if is_valid:
    print("\n🎉 Synthetic data generation completed successfully!")
    print("\nThe generated data includes:")
    print("• Evolved questions with IDs and evolution types")
    print("• Answers for each evolved question")
    print("• Relevant contexts for each question")
    print("\nAll three evolution types were implemented:")
    print("• Simple Evolution: Basic question refinement")
    print("• Multi-Context Evolution: Questions requiring multiple contexts")
    print("• Reasoning Evolution: Questions requiring complex reasoning")
else:
    print("\n❌ Validation failed. Please check the results.")


## Summary

This notebook successfully implements the optional assignment for Assignment 7 using LangGraph agents instead of the Knowledge Graph approach. The implementation includes:

### Key Features:
1. **LangGraph Agent Architecture**: Multi-agent system with specialized agents for different evolution types
2. **Evol Instruct Method**: Leverages evolutionary instruction generation for question enhancement
3. **Three Evolution Types**: Simple, Multi-Context, and Reasoning Evolution
4. **Structured Output**: Properly formatted results with question IDs, answers, and contexts

### Output Format:
- `List[dict]`: Evolved Questions with IDs and Evolution Types
- `List[dict]`: Question IDs and corresponding Answers
- `List[dict]`: Question IDs and relevant Contexts

### Graph Capabilities:
- **Simple Evolution**: Basic question refinement and enhancement
- **Multi-Context Evolution**: Questions requiring multiple document contexts
- **Reasoning Evolution**: Questions requiring complex reasoning and analysis

The implementation provides a robust alternative to the Knowledge Graph approach while maintaining the same output format and functionality requirements.
